# Sesión 1 · Qué es MCP y tu primer servidor remoto

**Curso MCP · servidores remotos** — notebook 1 de 4

Al final de esta sesión tendrás **tu propio servidor MCP desplegado en Cloud Run**, con URL
pública, exponiendo un dataset de BigQuery. Y habrás visto el protocolo por dentro, sin SDK
de por medio, antes de dejar que una librería te lo esconda.

| Bloque | Minutos |
|---|:--:|
| Qué es MCP y por qué existe | 20 |
| El protocolo por dentro | 25 |
| Despliegue a Cloud Run | 25 |
| Tools: servidor y cliente | 40 |

> **Antes de empezar** necesitas un proyecto de Google Cloud con facturación activada.
> Si llegas sin eso, esta sesión no te va a dar tiempo.

## 1. Qué es MCP y por qué existe

Tienes datos en BigQuery. Quieres que un modelo pueda consultarlos.

Sin un protocolo común, escribes un integrador para Claude, otro para ChatGPT, otro para
Cursor, otro para tu agente propio. Cuatro integraciones para el mismo dato, y cada vez que
cambia el esquema, cuatro sitios que tocar.

**Model Context Protocol es el contrato que rompe esa multiplicación.** Escribes un servidor
una vez, y cualquier cliente que hable MCP puede usarlo. La analogía útil: MCP es a los
modelos lo que un driver es a un sistema operativo.

### Las tres piezas

| Pieza | Qué es | En este curso |
|---|---|---|
| **Servidor** | Publica capacidades: acciones, datos, plantillas | Lo que vas a escribir, sobre BigQuery |
| **Cliente** | Habla el protocolo, uno por servidor conectado | El SDK de Python en este notebook |
| **Host** | La aplicación donde vive la conversación | Claude Desktop, ChatGPT, Cursor… |

El **host** es quien manda: decide qué servidores conecta, qué le enseña al modelo y qué
permisos concede. El servidor **ofrece**, nunca impone. Esa asimetría explica casi todas las
decisiones de diseño del protocolo, y conviene tenerla presente desde el principio.

### Por qué remoto y no local

Un servidor MCP puede correr como proceso local en la máquina del usuario. En este curso no
vamos a hacer eso **en ningún momento**, y no es un capricho:

- **Distribución.** Un servidor remoto se comparte con una URL. Uno local hay que instalarlo
  en cada máquina, con su runtime y sus dependencias.
- **Actualizaciones.** Despliegas una vez y todos tus usuarios tienen la versión nueva.
- **Control de acceso.** Es donde vive el OAuth de verdad, y donde tú decides quién ve qué.
- **Es donde están tus datos.** BigQuery ya está en la nube; bajar el servidor a un portátil
  para hablar con la nube es un rodeo.

Todo lo que aprendas aquí sirve tal cual en producción.

## 2. El protocolo por dentro

Antes de tocar el SDK, vamos a hablar con un servidor MCP **a mano**. Diez minutos de
incomodidad que ahorran meses de tratar la librería como una caja negra.

MCP es **JSON-RPC 2.0 sobre HTTP**. Nada más. Un `POST` con un cuerpo JSON, y una respuesta
JSON. La revisión que usamos es la `2026-07-28`.

In [ ]:
!pip install --quiet "mcp==2.0.0" httpx

In [ ]:
import httpx, json

# ← EDITAR: la URL del servidor de referencia que reparte quien imparte la sesión,
#           o la de tu propio servidor si ya has hecho el bloque 3.
URL = "https://PON-AQUI-LA-URL.run.app/mcp"

peticion = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "server/discover",
    "params": {
        "_meta": {
            # Los dos primeros son OBLIGATORIOS. Omitir cualquiera devuelve -32602.
            "io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": {},
            "io.modelcontextprotocol/clientInfo": {"name": "notebook-curso", "version": "1.0"},
        }
    },
}

CABECERAS = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    # La versión viaja DOS veces: en _meta y aquí. Sin esta cabecera el servidor
    # asume que hablas una revisión antigua y responde "Missing session ID".
    "MCP-Protocol-Version": "2026-07-28",
    # Obligatorias desde 2026-07-28. Mcp-Name identifica el recurso al que apuntas,
    # y va vacío cuando la petición no apunta a ninguno en concreto.
    "Mcp-Method": "server/discover",
    "Mcp-Name": "",
}

respuesta = httpx.post(URL, json=peticion, headers=CABECERAS, timeout=30)
print(respuesta.status_code)
print(json.dumps(respuesta.json(), indent=2, ensure_ascii=False)[:1200])

### Dos cosas que acabas de escribir y que parecen redundantes

**La versión del protocolo va dos veces**: en `_meta`, dentro del cuerpo, y en la cabecera
`MCP-Protocol-Version`. No es un descuido de la especificación. El cuerpo es la fuente de
verdad, pero la cabecera permite a proxies, balanceadores y al propio servidor enrutar la
petición **sin abrir el JSON**.

Y tiene una consecuencia muy práctica: **si te dejas la cabecera, el servidor da por hecho que
hablas una revisión antigua** y contesta `Bad Request: Missing session ID`, porque las
revisiones anteriores sí tenían sesiones. Un error desconcertante, cuya causa está en algo que
*no* escribiste.

**`clientCapabilities` es obligatorio aunque vaya vacío.** Un `{}` significa «no aporto nada
especial», que es una afirmación distinta de no decir nada. Si lo omites:

```
-32602  params._meta is missing the required envelope key(s):
        io.modelcontextprotocol/clientCapabilities
```

Los dos fallos son el mismo tipo de fallo: en un protocolo **stateless**, cada petición tiene
que explicarse entera. No hay un apretón de manos previo donde dejar dicho quién eres.

### Qué acabas de ver

`server/discover` es una llamada obligatoria del protocolo, y responde a todo lo que un
cliente necesita saber antes de empezar:

- **`supportedVersions`** — qué revisiones habla este servidor.
- **`capabilities`** — qué ofrece: tools, resources, prompts, extensiones.
- **`_meta.io.modelcontextprotocol/serverInfo`** — quién es.
- **`ttlMs` y `cacheScope`** — cuánto puede cachear el cliente esta respuesta.

Fíjate en lo que **no** hay: ninguna sesión, ningún identificador de conexión. La revisión
`2026-07-28` es **stateless**. Cada petición se explica sola y lleva su propia versión y sus
propias capacidades en `_meta`.

Eso tiene una consecuencia directa para nosotros: **sin estado de sesión, un servidor MCP
escala en horizontal sin afinidad de sesión**. Es exactamente lo que Cloud Run necesita para
repartir peticiones entre instancias sin pensárselo.

In [ ]:
# Un ayudante para no repetir el andamiaje en cada llamada.
# Fíjate en que _meta y las cabeceras se repiten SIEMPRE: no hay sesión donde
# dejar esa información guardada.
META = {
    "io.modelcontextprotocol/protocolVersion": "2026-07-28",
    "io.modelcontextprotocol/clientCapabilities": {},
}

def llamar(metodo, params=None, nombre=""):
    cuerpo = dict(params or {})
    cuerpo["_meta"] = META
    return httpx.post(
        URL,
        json={"jsonrpc": "2.0", "id": 1, "method": metodo, "params": cuerpo},
        headers={**CABECERAS, "Mcp-Method": metodo, "Mcp-Name": nombre},
        timeout=30,
    ).json()

respuesta = llamar("tools/list")["result"]
for t in respuesta["tools"]:
    print(f"- {t['name']}: {t.get('description','')[:70]}")

print("\nttlMs:", respuesta.get("ttlMs"), "· cacheScope:", respuesta.get("cacheScope"))

In [ ]:
# Y ahora una llamada de verdad. Ojo al `nombre`: tiene que coincidir con el
# `name` del cuerpo, o el servidor rechaza la petición.
r = llamar("tools/call", {"name": "listar_tablas", "arguments": {}}, nombre="listar_tablas")
print(r["result"]["content"][0]["text"])

### `Mcp-Name`: el error que te vas a encontrar

Esa cabecera **no identifica a tu cliente**, aunque lo parezca. Identifica **el recurso al
que apunta la petición**: el tool en un `tools/call`, el prompt en un `prompts/get`. Si no
coincide con el `name` que va en el cuerpo, el servidor corta:

```
-32020  mcp-name header does not match the request body's 'name' parameter
```

Ese `-32020` es de la familia de códigos que la revisión `2026-07-28` reservó para el
protocolo. Y la razón de que exista la cabecera es la misma que la de `MCP-Protocol-Version`:
**dejar que la infraestructura sepa a qué apunta la llamada sin abrir el cuerpo**. Un proxy
puede aplicar cuotas por herramienta, o enrutar, leyendo solo cabeceras.

> **Ejercicio (2 min).** Cambia `nombre="listar_tablas"` por `nombre="otra_cosa"` y observa
> el error. Luego prueba a quitar la cabecera `MCP-Protocol-Version` de `CABECERAS` y mira
> cómo cambia por completo el mensaje. Dos fallos que no vienen del código que has escrito,
> sino del que no has escrito.

> **Ejercicio (3 min).** Cambia `protocolVersion` a `"1999-01-01"` y vuelve a lanzar la
> celda. El servidor responde con `UnsupportedProtocolVersionError` y te dice qué versiones
> sí soporta. Así negocia un cliente que no sabe con quién habla.

## 3. Tu servidor en Cloud Run

Ahora el tuyo. Un único despliegue que ya contiene el código de las cuatro sesiones.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROYECTO = "codecrypto-ai"  # ← EDITAR
REGION = "europe-west1"
SERVICIO = "curso-mcp"

!gcloud config set project {PROYECTO}

In [ ]:
!gcloud services enable run.googleapis.com cloudbuild.googleapis.com bigquery.googleapis.com

In [ ]:
!git clone --quiet https://github.com/noelserdna/curso-mcp /content/curso-mcp
%cd /content/curso-mcp
!ls

In [ ]:
# Un único despliegue para todo el curso. Tarda un par de minutos la primera vez.
!gcloud run deploy {SERVICIO} \
  --source . \
  --region {REGION} \
  --allow-unauthenticated \
  --set-env-vars CURSO_MCP_DATASET=austin_bikeshare \
  --quiet

In [ ]:
MI_URL = !gcloud run services describe {SERVICIO} --region {REGION} --format="value(status.url)"
MI_URL = MI_URL[0] + "/mcp"
print("Tu servidor:", MI_URL)

> **`--allow-unauthenticated` es deliberado y temporal.** Ahora mismo cualquiera con la URL
> puede usar tu servidor y facturarte las consultas. En la sesión 3 lo cerramos. Que quede
> claro que un servidor abierto a internet **no** es un estado aceptable para algo que toca
> datos.

## 4. Tools: la primitiva central

Un **tool** es una acción que el modelo puede invocar. Es la primitiva que decide si tu
servidor sirve para algo.

Así se declara uno en el SDK de Python (esto ya está en `curso_mcp/server.py`):

```python
@mcp.tool(
    title="Describir tabla",
    description=(
        "Devuelve el esquema de una tabla: columnas, tipos y descripción. "
        "Úsalo antes de escribir una consulta para no inventar nombres de columna."
    ),
)
def describir_tabla(tabla: str) -> list[dict[str, Any]]:
    return bq.describir_tabla(tabla)
```

Tres cosas que hace el SDK por ti y conviene saber que hace:

1. **El esquema de entrada sale de las anotaciones de tipo.** `tabla: str` se convierte en
   un JSON Schema que el modelo lee para saber qué mandarte.
2. **El de salida, del tipo de retorno**, y el resultado viaja además como
   `structuredContent`, no solo como texto.
3. **Las excepciones se convierten en errores de protocolo** con `is_error`, en vez de tumbar
   la conexión.

In [ ]:
import asyncio
from mcp import Client

async def explorar():
    async with Client(MI_URL) as c:
        lista = await c.list_tools()
        for t in lista.tools:
            print(f"· {t.name}")
        print()
        r = await c.call_tool("listar_tablas", {})
        print(r.content[0].text)

await explorar()

> **Ojo con `await` en Colab.** El notebook ya corre dentro de un bucle de eventos, así que
> puedes usar `await` directamente en la celda. En un script normal harías
> `asyncio.run(explorar())`.

In [ ]:
async def esquema_de(tabla):
    async with Client(MI_URL) as c:
        r = await c.call_tool("describir_tabla", {"tabla": tabla})
        # El resultado estructurado, además del texto
        return r.structured_content

await esquema_de("bikeshare_trips")

## 5. Diseñar tools que el modelo entienda

Aquí se gana o se pierde la partida, y casi nada de ello es programar.

**El esquema de cada tool entra en el contexto del modelo en cada turno de conversación.**
No es documentación que se lee una vez: es texto que se paga una y otra vez, y sobre el que
el modelo decide a ciegas, antes de llamarte, si eres la herramienta adecuada.

Anthropic lo resume así: un tool es **un contrato entre un sistema determinista y un agente
que no lo es**. Se diseña asumiendo que quien lo va a usar puede malinterpretarlo.

> Los seis principios que siguen vienen de
> [*Writing effective tools for AI agents*](https://www.anthropic.com/engineering/writing-tools-for-agents)
> de Anthropic, aterrizados sobre nuestro servidor de BigQuery.

### Principio 1 · No envuelvas tu API: diseña para la tarea

El error más común es exponer un tool por cada endpoint que ya tienes. Eso traslada al modelo
el trabajo de recomponer la tarea, y cada paso intermedio cuesta una llamada y un montón de
contexto.

Mira nuestro propio servidor. Para responder «¿cuáles son las estaciones con más viajes?» el
modelo tiene que encadenar:

```
listar_tablas()            → ¿cuál de estas tablas tiene viajes?
describir_tabla("...")     → ¿cómo se llama la columna de estación?
describir_tabla("...")     → ¿y la de la otra tabla?
consultar("SELECT ...")    → por fin
```

Cuatro llamadas y cuatro respuestas ocupando contexto, para una pregunta que un analista
resolvería de una. La alternativa es un tool que **empaqueta la tarea**, no la API:

```python
@mcp.tool(
    title="Contexto del dataset",
    description=(
        "Devuelve de una vez el catálogo de tablas con sus columnas y una muestra de filas. "
        "Úsalo como primer paso ante cualquier pregunta sobre los datos: evita tener que "
        "encadenar listar_tablas y describir_tabla."
    ),
)
def contexto_dataset() -> dict:
    ...
```

**El criterio:** ¿estoy exponiendo lo que mi API hace, o lo que mi usuario necesita? Si los
tools se llaman siempre en la misma secuencia, esa secuencia debería ser un tool.

> Ojo, no es «cuantos menos, mejor». Es que la unidad correcta es **la tarea**, no el endpoint.

### Principio 2 · La descripción es el contrato

Escríbela **como se la explicarías a alguien que entra nuevo en el equipo**. Esa es la
comparación que usa Anthropic, y funciona: a un recién llegado le dices para qué sirve, cuándo
usarlo, qué devuelve y con qué no confundirlo.

Una descripción completa responde a cuatro cosas:

| | |
|---|---|
| **Qué hace** | «Devuelve el esquema de una tabla» |
| **Cuándo usarlo** | «Antes de escribir una consulta, para no inventar nombres de columna» |
| **Qué devuelve** | «Columnas, tipos y modo» |
| **Qué NO hace** | «No devuelve datos; para eso, `consultar`» |

Ese último punto es el que más llamadas equivocadas evita, y el que casi todo el mundo omite.

In [ ]:
# Vamos a medirlo. Dos versiones del mismo tool, y lo que le cuestan al modelo.
!pip install --quiet "mcp==2.0.0"

from mcp.server.mcpserver import MCPServer
import json, asyncio
from mcp import Client

pobre = MCPServer("pobre")
rico = MCPServer("rico")

@pobre.tool()
def get_data(t: str, l: int = 50) -> list:
    """Obtiene datos."""
    return []

@rico.tool(
    title="Consultar tabla",
    description=(
        "Devuelve las primeras filas de una tabla del dataset. "
        "Úsalo cuando ya sepas el nombre exacto de la tabla; si no lo sabes, llama antes a "
        "listar_tablas. No ejecuta SQL libre: para eso está consultar."
    ),
)
def leer_tabla(tabla: str, maximo_filas: int = 50) -> list:
    return []

async def esquema(servidor):
    async with Client(servidor) as c:
        return (await c.list_tools()).tools[0]

for nombre, servidor in [("POBRE", pobre), ("RICO", rico)]:
    t = await esquema(servidor)
    bruto = json.dumps({"name": t.name, "description": t.description,
                        "inputSchema": t.input_schema}, ensure_ascii=False)
    print(f"--- {nombre} · ~{len(bruto)//4} tokens por turno")
    print(bruto[:300], "\n")

Fíjate en el intercambio: la versión rica cuesta unas decenas de tokens más **en cada turno**.
A cambio evita llamadas equivocadas, que cuestan una ida y vuelta entera más la respuesta de
error. La descripción se amortiza sola.

Donde ese cálculo se invierte es con **treinta tools ricamente descritos**: ahí ya hablamos de
miles de tokens antes de que empiece la conversación, y toca replantear la superficie.

### Principio 3 · Nombres sin ambigüedad, en los tools y en los parámetros

Anthropic insiste en un detalle pequeño y muy rentable: **`user_id`, no `user`**. El modelo
tiene que adivinar si le pides un identificador, un nombre o un objeto entero, y acierta menos
de lo que crees.

| Ambiguo | Inequívoco | Por qué |
|---|---|---|
| `tabla` | `nombre_tabla` | ¿El nombre, o la tabla entera? |
| `limite` | `maximo_filas` | ¿Límite de qué: filas, bytes, segundos? |
| `q` | `consulta_sql` | Una letra no dice nada |
| `get_data` | `leer_tabla` | ¿Qué datos, de dónde? |

Y cuando tengas varios tools parecidos, **que cada descripción mencione a su hermano**:

```
listar_tablas   — ... Si ya sabes la tabla y quieres sus columnas, usa describir_tabla.
describir_tabla — ... No devuelve datos; para leer filas, usa consultar.
```

Con muchos tools, **agrúpalos por prefijo**: `bq_listar_tablas`, `bq_consultar`,
`drive_buscar`. Le da al modelo una pista de a qué mundo pertenece cada cosa.

### Principio 4 · Devuelve señal, no volcado

Lo que devuelve un tool también ocupa contexto. Y aquí hay un reflejo de programador que hace
daño: devolver el objeto entero de la API «por si acaso».

**Prefiere identificadores que signifiquen algo.** Un `uuid` de 36 caracteres es ruido que el
modelo no puede interpretar ni relacionar; un nombre sí.

| Devuelve | En vez de |
|---|---|
| `nombre_estacion` | `station_uuid` |
| `2026-03-14` | `1773446400000` |
| `"activa"` | `status_code: 3` |

Un patrón muy práctico del artículo de Anthropic: **un parámetro `formato` que module el
detalle**. En su medición, la versión concisa gastaba un 66 % menos de contexto que la
detallada.

In [ ]:
from typing import Literal

@rico.tool(
    title="Estaciones",
    description=(
        "Lista las estaciones del dataset. Usa formato='conciso' (por defecto) para explorar, "
        "y formato='detallado' solo si necesitas los identificadores técnicos para cruzar "
        "con otra tabla."
    ),
)
def listar_estaciones(formato: Literal["conciso", "detallado"] = "conciso") -> list[dict]:
    crudo = [
        {"station_id": "a3f9c1e2-7b44-4d10-9f2e-1c8b5d6a0e33",
         "name": "Congress & 8th", "status": 3, "lat": 30.267, "lon": -97.743,
         "created_ts": 1773446400000},
    ]
    if formato == "detallado":
        return crudo
    return [{"estacion": e["name"], "estado": "activa" if e["status"] == 3 else "inactiva"}
            for e in crudo]

conciso = json.dumps(listar_estaciones("conciso"), ensure_ascii=False)
detallado = json.dumps(listar_estaciones("detallado"), ensure_ascii=False)
print("conciso  :", conciso)
print("detallado:", detallado)
print(f"\nahorro: {100 - len(conciso)*100//len(detallado)}% de caracteres por fila")

Y cuando el resultado no cabe, **trunca diciéndolo**. Un volcado silenciosamente cortado hace
que el modelo saque conclusiones sobre datos que no ha visto:

```python
return {
    "filas": filas[:50],
    "aviso": f"Mostrando 50 de {total} filas. Afina la consulta con WHERE o usa LIMIT.",
}
```

Recuerda el límite real del que hablamos en la sesión 4: los hosts truncan alrededor de
150.000 caracteres, y Claude Code sobre 25.000 tokens. Un `SELECT *` sin `LIMIT` no llega
entero a ninguna parte.

### Principio 5 · Errores que enseñan

Un error es una oportunidad de guiar al modelo. La diferencia entre que se recupere solo o
entre en bucle está en si le dices **qué hacer a continuación**.

In [ ]:
TABLAS = ["bikeshare_trips", "bikeshare_stations"]

def error_pobre(tabla):
    return f"Error: {tabla} not found"

def error_util(tabla):
    from difflib import get_close_matches
    sugerencia = get_close_matches(tabla, TABLAS, n=1)
    extra = f" ¿Querías decir '{sugerencia[0]}'?" if sugerencia else ""
    return (f"La tabla '{tabla}' no existe en este dataset.{extra} "
            f"Tablas disponibles: {', '.join(TABLAS)}. "
            f"Usa listar_tablas para verlas con su descripción.")

print("POBRE:", error_pobre("bikeshare_trip"))
print()
print("ÚTIL :", error_util("bikeshare_trip"))

El segundo error contiene lo que el modelo necesita para arreglarlo **sin volver a preguntar
al usuario**: qué falló, la alternativa más probable, y qué opciones hay. Un error bien escrito
ahorra dos o tres turnos de conversación.

Lo mismo aplica a los errores de estrategia, no solo de dato:

```
"La consulta escanearía 8,2 GB, por encima del límite de 1 GB. Añade un filtro por fecha
 (la tabla está particionada por start_time) o agrega con COUNT en vez de traer filas."
```

### Principio 6 · Cuántos tools, y la línea que no se cruza

| Nº de tools | Qué hacer |
|---|---|
| 1–15 | Uno por acción. El punto dulce |
| 15–30 | Funciona, pero audita duplicados que puedan fusionarse |
| 30+ | Cambia a `search` + `execute`, y asciende los 3–5 más usados a tools propios |

Y dos reglas que en el directorio de conectores de Anthropic son **criterio de aprobación**,
no consejo:

**Separa lectura de escritura.** Un tool que según los argumentos consulta o borra se rechaza.
Que el modelo no pueda destruir nada por equivocarse de parámetro.

**Anota lo que hace cada tool.** `read_only_hint`, `destructive_hint` y `title` son lo que permite
al host conceder permisos automáticos a lo inofensivo y pedir confirmación para lo demás.

```python
from mcp_types import ToolAnnotations

@mcp.tool(
    title="Listar tablas",
    annotations=ToolAnnotations(read_only_hint=True, destructive_hint=False),
    description="...",
)
def listar_tablas() -> list[str]:
    ...
```

> Y una que sorprende: **la descripción no debe dar instrucciones de comportamiento al modelo**
> («llama siempre a este tool primero», «no uses la otra herramienta»). En la revisión del
> directorio eso se trata como inyección de prompt. Describe tu tool; no dirijas al agente.

### Cómo saber si lo has hecho bien

No se adivina: se mide. El método que propone Anthropic es sencillo y lo puedes montar en una
tarde.

**Escribe tareas realistas, no llamadas sueltas.** Una tarea buena necesita varias herramientas
y se parece a lo que pediría una persona:

> ✅ «¿Qué estaciones tuvieron más viajes el fin de semana pasado, y cómo se compara con la
> media del mes?»
>
> ❌ «Llama a consultar con este SQL.»

La segunda no evalúa tu diseño: ya le has dicho qué hacer.

**Mira las transcripciones.** Cuando el modelo se equivoca de tool, encadena llamadas
innecesarias o inventa un parámetro, casi nunca es culpa suya: es tu descripción. Anota qué
métricas mueves —número de llamadas por tarea, errores de parámetro, tokens consumidos— y
cambia una cosa cada vez.

### Repaso rápido

- [ ] ¿Expongo tareas o estoy calcando mi API?
- [ ] ¿Cada descripción dice qué hace, cuándo usarlo, qué devuelve y qué **no** hace?
- [ ] ¿Los nombres de parámetros son inequívocos (`nombre_tabla`, no `tabla`)?
- [ ] ¿Los tools parecidos se mencionan entre ellos?
- [ ] ¿Devuelvo identificadores legibles en vez de UUIDs?
- [ ] ¿Trunco diciendo que trunco?
- [ ] ¿Mis errores dicen qué hacer a continuación?
- [ ] ¿Lectura y escritura están en tools separados, con sus anotaciones?

**Para leer más:**
[Writing effective tools for AI agents](https://www.anthropic.com/engineering/writing-tools-for-agents) ·
[Criterios de revisión del directorio](https://claude.com/docs/connectors/building/review-criteria)

## Ejercicios

1. **Un tool nuevo.** Añade a `curso_mcp/server.py` un tool `contar_filas(tabla)` que
   devuelva el número de filas. Redespliega y llámalo desde aquí.
2. **Rompe una descripción.** Cambia la de `describir_tabla` por «Devuelve datos». Conecta
   Claude Desktop a tu URL y observa cómo empeoran sus decisiones sobre cuándo llamarlo.
3. **Sin SDK.** Reproduce la llamada a `listar_tablas` con `httpx` puro, como en el bloque 2.

## En la próxima sesión

Las otras primitivas: **resources** para datos que trae el host, **prompts** para workflows
enlatados, **interacción** para cuando el servidor necesita preguntarte algo a mitad de una
llamada, y **progreso** para operaciones que tardan.

## Limpieza: borrar lo que has creado

Un servicio de Cloud Run desplegado sigue existiendo hasta que lo borras, y el despliegue deja
más rastro del que parece:

| Artefacto | Qué es | ¿Cuesta dinero? |
|---|---|---|
| **Servicio de Cloud Run** | Tu servidor con su URL pública | Solo al recibir peticiones, pero **sigue accesible** |
| **Imágenes en Artifact Registry** | Cada despliegue sube una imagen nueva | Sí, por almacenamiento |
| **Bucket de staging de Cloud Build** | El código fuente que subiste | Sí, poco |

Lo que de verdad importa no es el coste, que es de céntimos: es que **mientras el servicio
exista con `--allow-unauthenticated`, cualquiera con la URL puede lanzarte consultas a
BigQuery**, y esas sí se facturan.

> ### ⚠️ El repositorio de imágenes es compartido
>
> `cloud-run-source-deploy` lo usan **todos** los servicios del proyecto desplegados con
> `--source`. Borrar el repositorio entero destruiría las imágenes de los demás. La celda de
> abajo borra **solo** la imagen de este curso. Nunca hagas
> `gcloud artifacts repositories delete cloud-run-source-deploy`.

In [ ]:
# Borra el servicio y la imagen de ESTE curso. No toca nada más del proyecto.
PROYECTO = "codecrypto-ai"   # ← EDITAR
REGION = "europe-west1"
SERVICIO = "curso-mcp"

# 1. El servicio: esto apaga la URL pública
!gcloud run services delete {SERVICIO} --project {PROYECTO} --region {REGION} --quiet

# 2. Solo la imagen de este servicio, no el repositorio que la contiene
!gcloud artifacts docker images delete \
  {REGION}-docker.pkg.dev/{PROYECTO}/cloud-run-source-deploy/{SERVICIO} \
  --project {PROYECTO} --delete-tags --quiet

In [ ]:
# Comprobación: el servicio ya no está y las imágenes de otros siguen ahí.
print("Servicios que quedan en la región:")
!gcloud run services list --project {PROYECTO} --region {REGION} --format="value(metadata.name)"

print("\nImágenes en el repositorio compartido (no debe salir curso-mcp):")
!gcloud artifacts docker images list \
  {REGION}-docker.pkg.dev/{PROYECTO}/cloud-run-source-deploy \
  --project {PROYECTO} --format="value(package)" 2>/dev/null | sed 's|.*/||' | sort -u

### Si además quieres vaciar el staging de Cloud Build

El bucket `{PROYECTO}_cloudbuild` guarda una copia del código de cada despliegue, del curso y
de cualquier otra cosa que hayas subido con `--source`. Revísalo antes de vaciarlo:

```bash
gsutil du -sh gs://{PROYECTO}_cloudbuild
gsutil ls gs://{PROYECTO}_cloudbuild/source | tail -5
```

Los objetos caducan solos si el bucket tiene regla de ciclo de vida. Si no la tiene, ponerla
es mejor idea que borrar a mano cada vez:

```bash
echo '{"rule":[{"action":{"type":"Delete"},"condition":{"age":30}}]}' > ciclo.json
gsutil lifecycle set ciclo.json gs://{PROYECTO}_cloudbuild
```

### Y si vuelves a necesitarlo

Redesplegar es una sola celda, la del bloque 3. Nada de lo que borras aquí es irrecuperable:
el código vive en el repositorio, no en Cloud Run.